In [ ]:
!pip install transformers
!pip install sentence-transformers
!pip install sentencepiece
!pip install datasets==3.6.0 # needs this version for self-instruct dataset
!pip install evaluate
!pip install nltk
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=47f0fe7d8cb978458a85e38a34631bf25667edb977995924b2bfbb67a8eb997c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

print(os.getcwd())

Mounted at /content/drive
/content


In [ ]:
import numpy
import torch
import scipy

import transformers
import sentence_transformers
import datasets
import evaluate
import nltk

import time

import random

import json

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
DEVICE = 'cpu'
if torch.cuda.is_available():
  DEVICE = 'cuda'
print(f'Device: {DEVICE}')

SEED = 42

random.seed(SEED)
numpy.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Device: cuda


In [ ]:
t5_large_name = 't5-large'

t5_large_model = transformers.T5ForConditionalGeneration.from_pretrained(t5_large_name)
t5_large_tokenizer = transformers.T5Tokenizer.from_pretrained(t5_large_name)

t5_large_model.to('cpu')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=4096, bias=False)
              (wo): Linear(in_features=4096, out_features=1024, bias=False)
              (d

In [ ]:
def drop_encoder_block(model, encoder_block_index):
    encoder_block = model.encoder.block[encoder_block_index]

    def hook_function(module, inputs, outputs):
      print(f'Drop Hook fired on Encoder Block: {encoder_block_index}')
      return (inputs[0],) + outputs[1:]

    return encoder_block.register_forward_hook(hook_function)


def drop_encoder_block_feed_forward_layer(model, encoder_block_index):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
      print(f'Drop Hook fired on Feed Forward Layer of Encoder Block: {encoder_block_index}')
      hidden_activations = inputs[0]
      return hidden_activations

    return feed_forward_layer.register_forward_hook(hook_function)


def mask_encoder_block(model, encoder_block_index, mask_threshold):
    encoder_block = model.encoder.block[encoder_block_index]

    def hook_function(module, inputs, outputs):
        print(f'Mask Hook fired on Encoder Block: {encoder_block_index}')

        hidden_activations = outputs[0]
        outputs_mask = (torch.rand_like(hidden_activations) > mask_threshold).float()
        masked_outputs = hidden_activations * outputs_mask

        return (masked_outputs,) + outputs[1:]

    return encoder_block.register_forward_hook(hook_function)


def mask_encoder_block_feed_forward_layer(model, encoder_block_index, mask_threshold):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
        print(f'Mask Hook fired on Feed Forward Layer of Encoder Block: {encoder_block_index}')

        hidden_activations = outputs
        outputs_mask = (torch.rand_like(hidden_activations) > mask_threshold).float()
        masked_outputs = hidden_activations * outputs_mask

        return masked_outputs

    return feed_forward_layer.register_forward_hook(hook_function)


def add_gaussian_noise_encoder_block(model, encoder_block_index, mean=0.0, standard_deviation=0.1):
    encoder_block = model.encoder.block[encoder_block_index]

    def hook_function(module, inputs, outputs):
        print(f'Gaussian Noise Hook fired on Encoder Block: {encoder_block_index}')

        hidden_activations = outputs[0]
        outputs_noise = mean + standard_deviation * torch.randn_like(hidden_activations)
        noised_outputs = hidden_activations + outputs_noise

        return (noised_outputs,) + outputs[1:]

    return encoder_block.register_forward_hook(hook_function)


def add_gaussian_noise_encoder_block_feed_forward_layer(model, encoder_block_index, mean=0.0, standard_deviation=0.1):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
        print(f'Gaussian Noise Hook fired on Feed Forward Layer of Encoder Block: {encoder_block_index}')

        hidden_activations = outputs
        outputs_noise = mean + standard_deviation * torch.randn_like(hidden_activations)
        noised_outputs = hidden_activations + outputs_noise

        return noised_outputs

    return feed_forward_layer.register_forward_hook(hook_function)


def drop_layer_normalization_encoder_block_self_attention_layer(model, encoder_block_index):
    layer_normalization = model.encoder.block[encoder_block_index].layer[0].layer_norm

    def hook_function(module, inputs, outputs):
        print(f'Drop Layer Normalization Hook fired on Self Attention Layer of Encoder Block: {encoder_block_index}')
        return inputs[0]

    return layer_normalization.register_forward_hook(hook_function)


def drop_layer_normalization_encoder_block_feed_forward_layer(model, encoder_block_index):
    layer_normalization = model.encoder.block[encoder_block_index].layer[1].layer_norm

    def hook_function(module, inputs, outputs):
        print(f'Drop Layer Normalization Hook fired on Feed Forward Layer of Encoder Block: {encoder_block_index}')
        return inputs[0]

    return layer_normalization.register_forward_hook(hook_function)


def mask_self_attention_heads_encoder_block(model, encoder_block_index, mask_threshold):
    self_attention_layer = model.encoder.block[encoder_block_index].layer[0].SelfAttention
    num_heads = self_attention_layer.n_heads
    head_dimension = model.config.d_model // num_heads

    def hook_function(module, inputs, outputs):
        print(f'Random Head Masking Hook fired in Encoder Block: {encoder_block_index}')
        hidden_states = outputs[0]
        batch_size, sequence_length, d_model = hidden_states.shape
        self_attention_outputs = hidden_states.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden_states.device) > mask_threshold).float()
        masked_outputs = self_attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return (masked_outputs.view(batch_size, sequence_length, d_model),) + outputs[1:]

    return self_attention_layer.register_forward_hook(hook_function)

In [ ]:
def drop_decoder_block(model, decoder_block_index):
    decoder_block = model.decoder.block[decoder_block_index]

    def hook_function(module, inputs, outputs):
        print(f'Drop Hook fired on Decoder Block: {decoder_block_index}')
        return (inputs[0],) + outputs[1:]

    return decoder_block.register_forward_hook(hook_function)


def drop_decoder_block_feed_forward_layer(model, decoder_block_index):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        print(f'Drop Hook fired on Feed Forward Layer of Decoder Block: {decoder_block_index}')
        return inputs[0]

    return feed_forward_layer.register_forward_hook(hook_function)


def mask_decoder_block(model, decoder_block_index, mask_threshold):
    decoder_block = model.decoder.block[decoder_block_index]

    def hook_function(module, inputs, outputs):
        print(f'Mask Hook fired on Decoder Block: {decoder_block_index}')

        hidden_activations = outputs[0]
        outputs_mask = (torch.rand_like(hidden_activations) > mask_threshold).float()
        masked_outputs = hidden_activations * outputs_mask

        return (masked_outputs,) + outputs[1:]

    return decoder_block.register_forward_hook(hook_function)


def mask_decoder_block_feed_forward_layer(model, decoder_block_index, mask_threshold):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        print(f'Mask Hook fired on Feed Forward Layer of Decoder Block: {decoder_block_index}')

        outputs_mask = (torch.rand_like(outputs) > mask_threshold).float()
        return outputs * outputs_mask

    return feed_forward_layer.register_forward_hook(hook_function)


def add_gaussian_noise_decoder_block(model, decoder_block_index, mean=0.0, standard_deviation=0.1):
    decoder_block = model.decoder.block[decoder_block_index]

    def hook_function(module, inputs, outputs):
        print(f'Gaussian Noise Hook fired on Decoder Block: {decoder_block_index}')

        hidden_activations = outputs[0]
        outputs_noise = mean + standard_deviation * torch.randn_like(hidden_activations)
        return (hidden_activations + outputs_noise,) + outputs[1:]

    return decoder_block.register_forward_hook(hook_function)


def add_gaussian_noise_decoder_block_feed_forward_layer(model, decoder_block_index, mean=0.0, standard_deviation=0.1):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        print(f'Gaussian Noise Hook fired on Feed Forward Layer of Decoder Block: {decoder_block_index}')
        outputs_noise = mean + standard_deviation * torch.randn_like(outputs)
        return outputs + outputs_noise

    return feed_forward_layer.register_forward_hook(hook_function)


def drop_layer_normalization_decoder_block_self_attention_layer(model, decoder_block_index):
    layer_normalization = model.decoder.block[decoder_block_index].layer[0].layer_norm

    def hook_function(module, inputs, outputs):
        print(f'Drop Layer Normalization Hook fired on Self Attention Layer of Decoder Block: {decoder_block_index}')
        return inputs[0]

    return layer_normalization.register_forward_hook(hook_function)


def drop_layer_normalization_decoder_block_feed_forward_layer(model, decoder_block_index):
    layer_normalization = model.decoder.block[decoder_block_index].layer[2].layer_norm

    def hook_function(module, inputs, outputs):
        print(f'Drop Layer Normalization Hook fired on Feed Forward Layer of Decoder Block: {decoder_block_index}')
        return inputs[0]

    return layer_normalization.register_forward_hook(hook_function)


def mask_self_attention_heads_decoder_block(model, decoder_block_index, mask_threshold):
    self_attention_layer = model.decoder.block[decoder_block_index].layer[0].SelfAttention
    num_heads = self_attention_layer.n_heads
    head_dimension = model.config.d_model // num_heads

    def hook_function(module, inputs, outputs):
        print(f'Random Self Attention Head Masking Hook fired in Decoder Block: {decoder_block_index}')
        hidden_states = outputs[0]
        batch_size, sequence_length, d_model = hidden_states.shape
        self_attention_outputs = hidden_states.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden_states.device) > mask_threshold).float()
        masked_outputs = self_attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return (masked_outputs.view(batch_size, sequence_length, d_model),) + outputs[1:]

    return self_attention_layer.register_forward_hook(hook_function)


def mask_cross_attention_heads_decoder_block(model, decoder_block_index, mask_threshold):
    cross_attention_layer = model.decoder.block[decoder_block_index].layer[1].EncDecAttention
    num_heads = cross_attention_layer.n_heads
    head_dimension = model.config.d_model // num_heads

    def hook_function(module, inputs, outputs):
        print(f'Random Cross Attention Head Masking Hook fired in Decoder Block: {decoder_block_index}')
        hidden_states = outputs[0]
        batch_size, sequence_length, d_model = hidden_states.shape
        cross_attention_outputs = hidden_states.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden_states.device) > mask_threshold).float()
        masked_outputs = cross_attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return (masked_outputs.view(batch_size, sequence_length, d_model),) + outputs[1:]

    return cross_attention_layer.register_forward_hook(hook_function)

In [ ]:
databricks_dolly_15k_name = 'databricks/databricks-dolly-15k'

databricks_dolly_15k_dataset = datasets.load_dataset(databricks_dolly_15k_name)

In [ ]:
def preprocess_databricks_dolly_15k_dataset(dataset):

  def filter_sample(sample):
    if sample['category'] == 'classification':
        return sample['context'] != ''
    return sample['category'] in ['closed_qa', 'information_extraction', 'summarization']

  def preprocess_sample(sample):
    instruction = sample.get('instruction', '').strip()
    context = sample.get('context', '').strip()
    category = sample.get('category', '')

    if category in ['closed_qa', 'information_extraction']:
        sample['input'] = f'question: {instruction} context: {context}'
    elif category == 'summarization':
        text_to_summarize = f'{instruction} {context}'.strip()
        sample['input'] = f'summarize: {text_to_summarize}'
    elif category == 'classification':
        if context:
            sample['input'] = f'question: {instruction} context: {context}'
        else:
            sample['input'] = instruction

    sample['input'] = ' '.join(sample['input'].split())
    sample['output'] = sample.get('response', '').strip()

    return sample

  dataset = dataset.filter(filter_sample)
  dataset = dataset.map(preprocess_sample)

  dataset = dataset.remove_columns(
      ['instruction', 'context', 'response', 'category']
  )

  TRAIN_WEIGHT = 10
  VALIDATION_WEIGHT = 1
  TEST_WEIGHT = 4

  dataset_split_0 = dataset['train'].train_test_split(
      train_size=TRAIN_WEIGHT / (TRAIN_WEIGHT + VALIDATION_WEIGHT + TEST_WEIGHT),
      test_size=(VALIDATION_WEIGHT + TEST_WEIGHT) / (TRAIN_WEIGHT + VALIDATION_WEIGHT + TEST_WEIGHT),
      seed=SEED
  )
  dataset_split_1 = dataset_split_0['test'].train_test_split(
      train_size=VALIDATION_WEIGHT / (VALIDATION_WEIGHT + TEST_WEIGHT),
      test_size=TEST_WEIGHT / (VALIDATION_WEIGHT + TEST_WEIGHT),
      seed=SEED
  )

  dataset = datasets.DatasetDict({
      'train': dataset_split_0['train'],
      'validation': dataset_split_1['train'],
      'test': dataset_split_1['test']
  })

  return dataset

In [ ]:
databricks_dolly_15k_dataset = preprocess_databricks_dolly_15k_dataset(databricks_dolly_15k_dataset)

In [ ]:
qwen3_embedding_06b_name = 'Qwen/Qwen3-Embedding-0.6B'

qwen3_embedding_06b_model = transformers.AutoModel.from_pretrained(qwen3_embedding_06b_name)
qwen3_embedding_06b_tokenizer = transformers.AutoTokenizer.from_pretrained(qwen3_embedding_06b_name)

qwen3_embedding_06b_model.to('cpu')

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [ ]:
def obtain_model_generated_outputs(model, tokenizer, dataset, dataset_type='train', MAX_NUM_SAMPLES=128):

  def collate_function(batch):
    input_texts = [sample['input'] for sample in batch]

    tokens = tokenizer(input_texts, return_tensors='pt', padding=True, truncation=True, max_length=128)

    return tokens

  tokenizer.padding_side = 'right'

  dataset_subset = dataset[dataset_type].select(range(min(len(dataset[dataset_type]), MAX_NUM_SAMPLES)))

  data_loader = torch.utils.data.DataLoader(
      dataset_subset,
      batch_size=64,
      shuffle=False,
      collate_fn=collate_function
  )

  generated_outputs = []

  model.to(DEVICE)
  model.eval()
  for batch_index, batch in enumerate(data_loader):
    batch = {key: value.to(DEVICE) for key, value in batch.items()}

    with torch.no_grad():
      output_tokens = model.generate(
          **batch,
          max_new_tokens=256,
          do_sample=False, # no randomness (no sampling from distribution)
          num_beams=1, # no multiple possible continuations in parallel

          # no_repeat_ngram_size=4,
          # repetition_penalty=1.25,
          # encoder_no_repeat_ngram_size=4,
          # length_penalty=1.0,
          # min_length=5,
      )

    decoded_outputs = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)

    generated_outputs.extend(decoded_outputs)

    # if (batch_index + 1) % 50 == 0:
    print(f'Batch {batch_index + 1}/{len(data_loader)} Finished')
  model.to('cpu')

  if 'generated_output' in dataset_subset.column_names:
    dataset_subset = dataset_subset.remove_columns('generated_output')
  dataset_subset = dataset_subset.add_column('generated_output', generated_outputs)

  return dataset_subset

In [ ]:
def evaluate_model_generative_task(model, tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  outputs = dataset_subset['output']
  generated_outputs = dataset_subset['generated_output']

  tokenizer.padding_side = 'right'

  batch_size = 64
  similarities = []

  num_samples = len(outputs)

  model.to(DEVICE)
  model.eval()
  for index in range(0, num_samples, batch_size):
    batch_outputs = outputs[index:min(index + batch_size, num_samples)]
    batch_generated_outputs = generated_outputs[index:min(index + batch_size, num_samples)]

    tokenized_outputs = tokenizer(batch_outputs, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
    tokenized_generated_outputs = tokenizer(batch_generated_outputs, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)

    with torch.no_grad():
      model_outputs = model(**tokenized_outputs)
      model_generated_outputs = model(**tokenized_generated_outputs)

      last_token_index_outputs = (tokenized_outputs['attention_mask'].sum(dim=1) - 1)
      last_token_index_generated_outputs = (tokenized_generated_outputs['attention_mask'].sum(dim=1) - 1)

      embedded_outputs = model_outputs.last_hidden_state[torch.arange(last_token_index_outputs.shape[0]), last_token_index_outputs]
      embedded_generated_outputs = model_generated_outputs.last_hidden_state[torch.arange(last_token_index_generated_outputs.shape[0]), last_token_index_generated_outputs]

      embedded_outputs = torch.nn.functional.normalize(embedded_outputs, p=2, dim=1)
      embedded_generated_outputs = torch.nn.functional.normalize(embedded_generated_outputs, p=2, dim=1)

      batch_similarities = (embedded_outputs * embedded_generated_outputs).sum(dim=1).cpu().tolist()
      similarities.extend(batch_similarities)

    print(f'Sample {index + batch_size}/{num_samples} Finished')

  model.to('cpu')
  torch.cuda.empty_cache()

  results = dict()
  results['similarities'] = similarities
  results['similarity_mean'] = sum(results['similarities']) / len(results['similarities'])

  if 'rouge' in metric_names:
    rouge_metric = evaluate.load('rouge')
    results['rouge'] = rouge_metric.compute(predictions=generated_outputs, references=outputs)

  if 'meteor' in metric_names:
    meteor_metric = evaluate.load('meteor')
    results['meteor_mean'] = meteor_metric.compute(predictions=generated_outputs, references=outputs)['meteor']

    results['meteor'] = []

    for index in range(num_samples):
      results['meteor'].append(meteor_metric.compute(predictions=[generated_outputs[index]], references=[outputs[index]])['meteor'])

  return results

In [ ]:
databricks_dolly_15k_subset = obtain_model_generated_outputs(t5_large_model, t5_large_tokenizer, databricks_dolly_15k_dataset, 'test')
databricks_dolly_15k_results = evaluate_model_generative_task(qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)

Batch 1/2 Finished
Batch 2/2 Finished


Flattening the indices:   0%|          | 0/128 [00:00<?, ? examples/s]

Sample 64/128 Finished
Sample 128/128 Finished


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
print(numpy.mean(databricks_dolly_15k_results['similarities']))

print(databricks_dolly_15k_subset[16]['input'])
print('\n\n\n')
print(databricks_dolly_15k_subset[16]['generated_output'])
print('\n\n\n')
print(databricks_dolly_15k_subset[16]['output'])

0.6252508163452148
question: Given this reference text, give me a list of all the schools attended by Tripp. context: Tripp attended The Hill School at Pottstown, Pennsylvania where he drew his attention for his talent as a football player. Tripp enrolled at the University of Chicago and played at the tackle for Amos Alonzo Stagg's 1902 football team. In January 1903, Tripp transferred to Yale University, where he played guard for Yale University's football teams in 1904 and 1905 after sitting out the 1903 season. He was captain of Yale's championship team of 1905, and was selected by Walter Camp as a first-team All-American in 1905. He later worked as a stockbroker He died in October 1962.




University of Chicago




The Hill School, University of Chicago, Yale University


In [ ]:
del databricks_dolly_15k_results
del databricks_dolly_15k_subset
del databricks_dolly_15k_dataset

In [ ]:
self_instruct_name = 'yizhongw/self_instruct'

self_instruct_dataset = datasets.load_dataset(self_instruct_name, trust_remote_code=True)

README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

self_instruct.py:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/82612 [00:00<?, ? examples/s]

In [ ]:
def preprocess_self_instruct_dataset(dataset):

  def preprocess_sample(sample):
      prompt = sample.get('prompt', '').strip()

      sample['input'] = f'instruction: {prompt}'
      sample['input'] = ' '.join(sample['input'].split())

      sample['output'] = sample.get('completion', '').strip()

      return sample

  dataset = dataset.map(preprocess_sample)
  dataset = dataset.remove_columns(
      ['prompt', 'completion']
  )

  TRAIN_WEIGHT = 56000
  VALIDATION_WEIGHT = 8000
  TEST_WEIGHT = 16000

  dataset_split_0 = dataset['train'].train_test_split(
      train_size=TRAIN_WEIGHT / (TRAIN_WEIGHT + VALIDATION_WEIGHT + TEST_WEIGHT),
      test_size=(VALIDATION_WEIGHT + TEST_WEIGHT) / (TRAIN_WEIGHT + VALIDATION_WEIGHT + TEST_WEIGHT),
      seed=SEED
  )
  dataset_split_1 = dataset_split_0['test'].train_test_split(
      train_size=VALIDATION_WEIGHT / (VALIDATION_WEIGHT + TEST_WEIGHT),
      test_size=TEST_WEIGHT / (VALIDATION_WEIGHT + TEST_WEIGHT),
      seed=SEED
  )

  dataset = datasets.DatasetDict({
      'train': dataset_split_0['train'],
      'validation': dataset_split_1['train'],
      'test': dataset_split_1['test']
  })

  return dataset

In [ ]:
self_instruct_dataset = preprocess_self_instruct_dataset(self_instruct_dataset)

Map:   0%|          | 0/82612 [00:00<?, ? examples/s]

In [ ]:
self_instruct_subset = obtain_model_generated_outputs(t5_large_model, t5_large_tokenizer, self_instruct_dataset, 'test')
self_instruct_results = evaluate_model_generative_task(qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, self_instruct_subset)

Batch 1/8 Finished
Batch 2/8 Finished
Batch 3/8 Finished
Batch 4/8 Finished
Batch 5/8 Finished
Batch 6/8 Finished
Batch 7/8 Finished
Batch 8/8 Finished


Flattening the indices:   0%|          | 0/512 [00:00<?, ? examples/s]

Sample 64/512 Finished
Sample 128/512 Finished
Sample 192/512 Finished
Sample 256/512 Finished
Sample 320/512 Finished
Sample 384/512 Finished
Sample 448/512 Finished
Sample 512/512 Finished


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
print(numpy.mean(self_instruct_results['similarities']))

print(self_instruct_subset[10]['input'])
print('\n\n\n')
print(self_instruct_subset[10]['generated_output'])
print('\n\n\n')
print(self_instruct_subset[10]['output'])

0.5435161590576172
instruction: You have n balls numbered 1 through n. you need to arrange them in order without removing or swapping any ball. n = 3, balls = [1, 2, 3] Output:




: You have n balls numbered 1 through n. you need to arrange them in order without removing any ball.: You have n balls numbered 1 through n. you need to arrange them in order without removing any ball.::n n through n... balls n ballsn. n = 3, balls = [1, 2, 3] n..:




[3, 2, 1]


In [ ]:
del self_instruct_results
del self_instruct_subset
del self_instruct_dataset

In [ ]:
cnn_dailymail_name = 'abisee/cnn_dailymail'

cnn_dailymail_dataset = datasets.load_dataset(cnn_dailymail_name, '3.0.0')

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
def preprocess_cnn_dailymail_dataset(dataset):

  def preprocess_sample(sample):
      article = sample.get('article', '').strip()

      sample['input'] = f'summarize: {article}'
      sample['input'] = ' '.join(sample['input'].split())

      sample['output'] = sample.get('highlights', '').strip()

      return sample

  dataset = dataset.map(preprocess_sample)
  dataset = dataset.remove_columns(
      ['article', 'highlights', 'id']
  )

  return dataset

In [ ]:
cnn_dailymail_dataset = preprocess_cnn_dailymail_dataset(cnn_dailymail_dataset)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
cnn_dailymail_subset = obtain_model_generated_outputs(t5_large_model, t5_large_tokenizer, cnn_dailymail_dataset, 'test')
cnn_dailymail_results = evaluate_model_generative_task(qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, cnn_dailymail_subset)

Batch 1/8 Finished
Batch 2/8 Finished
Batch 3/8 Finished
Batch 4/8 Finished
Batch 5/8 Finished
Batch 6/8 Finished
Batch 7/8 Finished
Batch 8/8 Finished
Sample 64/512 Finished
Sample 128/512 Finished
Sample 192/512 Finished
Sample 256/512 Finished
Sample 320/512 Finished
Sample 384/512 Finished
Sample 448/512 Finished
Sample 512/512 Finished


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
print(numpy.mean(cnn_dailymail_results['similarities']))

print(cnn_dailymail_subset[7]['input'])
print('\n\n\n')
print(cnn_dailymail_subset[7]['generated_output'])
print('\n\n\n')
print(cnn_dailymail_subset[7]['output'])

0.6475925445556641
summarize: (CNN)Andrew Getty, one of the heirs to billions of oil money, appears to have died of natural causes, a Los Angeles Police Department spokesman said. The coroner's preliminary assessment is there was no foul play involved in the death of Getty, grandson of oil tycoon J. Paul Getty, said Detective Meghan Aguilar. Andrew Getty, 47, had "several health issues," Aguilar said, adding that an autopsy will be conducted. There is no criminal investigation underway, he said. Some medication had also been recovered from Getty's home, though investigators don't know whether Getty was taking it or what his medical history was, Ed Winter, assistant chief in the Los Angeles County coroner's office, told CNN affiliate KTLA Tuesday night. KTLA reported that Getty was found on his side near a bathroom in his home. Getty's parents, Ann and Gordon Getty, released a statement confirming their son's death and asking for privacy. Where the Getty family fortune came from . Gordo

In [ ]:
del cnn_dailymail_results
del cnn_dailymail_subset
del cnn_dailymail_dataset

In [ ]:
def obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset):

  def collate_function(batch):
    input_texts = [sample['input'] for sample in batch]

    tokens = tokenizer(input_texts, return_tensors='pt', padding=True, truncation=True, max_length=128)

    return tokens

  tokenizer.padding_side = 'right'

  data_loader = torch.utils.data.DataLoader(
      dataset_subset,
      batch_size=64,
      shuffle=False,
      collate_fn=collate_function
  )

  generated_outputs = []

  model.to(DEVICE)
  model.eval()
  for batch_index, batch in enumerate(data_loader):
    batch = {key: value.to(DEVICE) for key, value in batch.items()}

    with torch.no_grad():
      output_tokens = model.generate(
          **batch,
          max_new_tokens=256,
          do_sample=False, # no randomness (no sampling from distribution)
          num_beams=1, # no multiple possible continuations in parallel

          # no_repeat_ngram_size=4,
          # repetition_penalty=1.25,
          # encoder_no_repeat_ngram_size=4,
          # length_penalty=1.0,
          # min_length=5,
      )

    decoded_outputs = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)

    generated_outputs.extend(decoded_outputs)

    # if (batch_index + 1) % 50 == 0:
    print(f'Batch {batch_index + 1}/{len(data_loader)} Finished')
  model.to('cpu')

  if 'generated_output' in dataset_subset.column_names:
    dataset_subset = dataset_subset.remove_columns('generated_output')
  dataset_subset = dataset_subset.add_column('generated_output', generated_outputs)

  return dataset_subset

In [ ]:
def save_results(save_path, results):
  with open(save_path, 'w') as results_file:
    json.dump(results, results_file)


def component_dropping_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/component_dropping_evaluation.json'

  results = dict()

  # Drop Encoder and Decoder Blocks, only one by one
  # First Encoder Blocks
  for encoder_block_index in range(NUM_ENCODER_BLOCKS):
    encoder_block_hook = drop_encoder_block(model, encoder_block_index)

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    encoder_block_hook.remove()

    results[f'drop_encoder_block_{encoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  print('Finished Single Encoder Blocks')

  # Decoder Blocks
  for decoder_block_index in range(NUM_DECODER_BLOCKS):
    decoder_block_hook = drop_decoder_block(model, decoder_block_index)

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    decoder_block_hook.remove()

    results[f'drop_decoder_block_{decoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  print('Finished Single Decoder Blocks')

  # Drop Encoder Blocks starting from beggining to end
  encoder_block_hooks = []
  for encoder_block_index in range(NUM_ENCODER_BLOCKS - 1):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'drop_encoder_block_from_beggining_{encoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  print('Finished Encoder Blocks starting from beggining')

  # Drop Decoder Blocks starting from beggining
  decoder_block_hooks = []
  for decoder_block_index in range(NUM_DECODER_BLOCKS - 1):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'drop_decoder_block_from_beggining_{decoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  print('Finished Decoder Blocks starting from beggining')

  # Drop Encoder Block starting from end
  encoder_block_hooks = []
  for encoder_block_index in range(NUM_ENCODER_BLOCKS - 1, 0, -1):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'drop_encoder_block_from_end_{encoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  print('Finished Encoder Blocks starting from end')

  # Drop Decoder Blocks starting from end
  decoder_block_hooks = []
  for decoder_block_index in range(NUM_DECODER_BLOCKS - 1, 0, -1):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'drop_decoder_block_from_end_{decoder_block_index}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  print('Finished Decoder Blocks starting from end')

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
def component_dropping_odd_even_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/component_dropping_odd_even_evaluation.json'

  results = dict()

  # Drop all Encoder Even Blocks
  encoder_block_hooks = []
  for encoder_block_index in range(0, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_encoder_block_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  # Drop all Decoder Even Blocks
  decoder_block_hooks = []
  for decoder_block_index in range(0, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_decoder_block_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Encoder Odd Blocks
  encoder_block_hooks = []
  for encoder_block_index in range(1, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_encoder_block_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  # Drop all Decoder Odd Blocks
  decoder_block_hooks = []
  for decoder_block_index in range(1, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_decoder_block_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Even Blocks
  encoder_block_hooks = []
  for encoder_block_index in range(0, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))
  decoder_block_hooks = []
  for decoder_block_index in range(0, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_blocks_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()
  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Odd Blocks
  encoder_block_hooks = []
  for encoder_block_index in range(1, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block(model, encoder_block_index))
  decoder_block_hooks = []
  for decoder_block_index in range(1, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_blocks_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()
  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
def feed_forward_layer_dropping_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/feed_forward_layer_dropping_evaluation.json'

  results = dict()

  # Drop all Encoder Even Feed Forward
  encoder_block_hooks = []
  for encoder_block_index in range(0, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_encoder_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  # Drop all Decoder Even Feed Forward
  decoder_block_hooks = []
  for decoder_block_index in range(0, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_decoder_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Encoder Odd Feed Forward
  encoder_block_hooks = []
  for encoder_block_index in range(1, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_encoder_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  # Drop all Decoder Odd Feed Forward
  decoder_block_hooks = []
  for decoder_block_index in range(1, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_decoder_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Even Feed Forward
  encoder_block_hooks = []
  for encoder_block_index in range(0, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))
  decoder_block_hooks = []
  for decoder_block_index in range(0, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_all_even'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()
  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Odd Feed Forward
  encoder_block_hooks = []
  for encoder_block_index in range(1, NUM_ENCODER_BLOCKS, 2):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))
  decoder_block_hooks = []
  for decoder_block_index in range(1, NUM_DECODER_BLOCKS, 2):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_all_odd'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()
  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Feed Forward Encoder
  encoder_block_hooks = []
  for encoder_block_index in range(NUM_ENCODER_BLOCKS):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_encoder_all'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()

  # Drop all Feed Forward Decoder
  decoder_block_hooks = []
  for decoder_block_index in range(NUM_DECODER_BLOCKS):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_decoder_all'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  # Drop all Feed Forward
  encoder_block_hooks = []
  for encoder_block_index in range(NUM_ENCODER_BLOCKS):
    encoder_block_hooks.append(drop_encoder_block_feed_forward_layer(model, encoder_block_index))
  decoder_block_hooks = []
  for decoder_block_index in range(NUM_DECODER_BLOCKS):
    decoder_block_hooks.append(drop_decoder_block_feed_forward_layer(model, decoder_block_index))

  dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
  current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

  results[f'drop_feed_forward_all'] = current_results
  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  for encoder_block_hook in encoder_block_hooks:
    encoder_block_hook.remove()
  for decoder_block_hook in decoder_block_hooks:
    decoder_block_hook.remove()

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
def attention_masking_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/attention_masking_evaluation.json'

  results = dict()

  masking_thresholds = [0.1, 0.25, 0.5, 0.75]

  for masking_threshold in masking_thresholds:

    # Mask all Self Attention in Encoder
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_self_attention_heads_encoder_block(model, encoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_all_self_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()

    # Mask all Self Attention in Decoder
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_self_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_decoder_all_self_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask all Cross Attention in Decoder
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_cross_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_decoder_all_cross_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask all Self Attention + Cross Attention in Decoder
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_self_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))
      decoder_block_hooks.append(mask_cross_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_decoder_all_self_cross_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask all Self Attention Encoder + Decoder
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_self_attention_heads_encoder_block(model, encoder_block_index, masking_threshold))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_self_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_decoder_all_self_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask Self Attention (both encoder + decoder) and decoder cross
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_self_attention_heads_encoder_block(model, encoder_block_index, masking_threshold))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_self_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))
      decoder_block_hooks.append(mask_cross_attention_heads_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_decoder_all_self_cross_attention_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
def gaussian_noising_component_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):

  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/gaussian_noising_component_evaluation.json'

  results = dict()

  standard_deviations = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

  for standard_deviation in standard_deviations:

    # Add Noise on Encoders
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(add_gaussian_noise_encoder_block(model, encoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_encoder_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()

    # Add Noise on Decoders
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(add_gaussian_noise_decoder_block(model, decoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_decoder_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Add Noise on Both
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(add_gaussian_noise_encoder_block(model, encoder_block_index, mean=0.0, standard_deviation=standard_deviation))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(add_gaussian_noise_decoder_block(model, decoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Add Noise on Encoder Feed Forwards
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(add_gaussian_noise_encoder_block_feed_forward_layer(model, encoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_encoder_feed_forward_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()

    # Add Noise on Decoder Feed Forwards
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(add_gaussian_noise_decoder_block_feed_forward_layer(model, decoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_decoder_feed_forward_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Add Noise on all feed forwards
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(add_gaussian_noise_encoder_block_feed_forward_layer(model, encoder_block_index, mean=0.0, standard_deviation=standard_deviation))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(add_gaussian_noise_decoder_block_feed_forward_layer(model, decoder_block_index, mean=0.0, standard_deviation=standard_deviation))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'add_gaussian_noise_feed_forward_all_{standard_deviation}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
def feed_forward_encoder_decoder_masking_evaluation(model, tokenizer, similarity_model, similarity_tokenizer, dataset_subset, metric_names=['rouge', 'meteor']):
  NUM_ENCODER_BLOCKS = 24
  NUM_DECODER_BLOCKS = 24

  GOOGLE_DRIVE_SAVE_PATH = '/content/drive/My Drive/feed_forward_encoder_decoder_masking_evaluation.json'

  results = dict()

  masking_thresholds = [0.1, 0.2, 0.25, 0.5, 0.75, 0.8]

  for masking_threshold in masking_thresholds:

    # Mask all feed forward Encoder
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_encoder_block_feed_forward_layer(model, encoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_feed_forward_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()

    # Mask all feed forward Decoder
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_decoder_block_feed_forward_layer(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_decoder_feed_forward_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask all feed forwards
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_encoder_block_feed_forward_layer(model, encoder_block_index, masking_threshold))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_decoder_block_feed_forward_layer(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_feed_forward_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask Encoder Blocks entirely
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_encoder_block(model, encoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()

    # Mask Decoder Blocks entirely
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_decoder_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

    # Mask all blocks entirely
    encoder_block_hooks = []
    for encoder_block_index in range(NUM_ENCODER_BLOCKS):
      encoder_block_hooks.append(mask_encoder_block(model, encoder_block_index, masking_threshold))
    decoder_block_hooks = []
    for decoder_block_index in range(NUM_DECODER_BLOCKS):
      decoder_block_hooks.append(mask_decoder_block(model, decoder_block_index, masking_threshold))

    dataset_subset = obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset)
    current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, dataset_subset, metric_names)

    results[f'mask_encoder_decoder_all_{masking_threshold}'] = current_results
    save_results(GOOGLE_DRIVE_SAVE_PATH, results)

    for encoder_block_hook in encoder_block_hooks:
      encoder_block_hook.remove()
    for decoder_block_hook in decoder_block_hooks:
      decoder_block_hook.remove()

  save_results(GOOGLE_DRIVE_SAVE_PATH, results)

  return results

In [ ]:
'''
ALREADY_EXISTING_RESULTS_PATH = '/content/drive/My Drive/component_dropping_evaluation_uncomplete.json'

with open(ALREADY_EXISTING_RESULTS_PATH, 'r') as file:
    already_existing_results = json.load(file)

print(already_existing_results['drop_decoder_block_from_end_19'])
'''

{'similarities': [0.546875, 0.953125, 0.287109375, 0.79296875, 0.6484375, 0.2470703125, 0.53515625, 0.54296875, 0.330078125, 0.30078125, 0.5546875, 0.56640625, 0.7734375, 0.421875, 0.263671875, 0.1962890625, 0.453125, 0.267578125, 0.25390625, 0.6328125, 0.6015625, 0.466796875, 0.46484375, 0.66015625, 0.29296875, 0.65625, 0.345703125, 0.3359375, 0.7890625, 0.4921875, 0.376953125, 0.1845703125, 0.55859375, 0.337890625, 0.28515625, 0.70703125, 0.5859375, 0.4453125, 0.62890625, 0.70703125, 0.4140625, 0.30078125, 0.462890625, 0.65625, 0.31640625, 0.2578125, 0.44140625, 0.36328125, 0.43359375, 0.6484375, 0.4296875, 0.703125, 0.357421875, 0.45703125, 0.4765625, 0.34765625, 0.5703125, 0.77734375, 0.35546875, 0.384765625, 0.74609375, 0.357421875, 0.37109375, 0.59375, 1.0078125, 0.5, 0.8203125, 0.458984375, 0.5078125, 0.578125, 0.298828125, 0.640625, 0.99609375, 0.294921875, 0.46484375, 0.57421875, 0.99609375, 0.71484375, 0.42578125, 0.7734375, 0.58984375, 0.26953125, 0.458984375, 0.609375, 0.55

In [ ]:
databricks_dolly_15k_subset = obtain_model_generated_outputs(t5_large_model, t5_large_tokenizer, databricks_dolly_15k_dataset, 'test')

component_dropping_results = component_dropping_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)
# component_dropping_odd_even_results = component_dropping_odd_even_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)
# feed_forward_layer_dropping_results = feed_forward_layer_dropping_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)

# attention_masking_results = attention_masking_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)

# gaussian_noising_component_results = gaussian_noising_component_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)
# feed_forward_encoder_decoder_masking_results = feed_forward_encoder_decoder_masking_evaluation(t5_large_model, t5_large_tokenizer, qwen3_embedding_06b_model, qwen3_embedding_06b_tokenizer, databricks_dolly_15k_subset)

Streaming output truncated to the last 5000 lines.
Mask Hook fired on Feed Forward Layer of Decoder Block: 20
Mask Hook fired on Feed Forward Layer of Decoder Block: 21
Mask Hook fired on Feed Forward Layer of Decoder Block: 22
Mask Hook fired on Feed Forward Layer of Decoder Block: 23
Mask Hook fired on Feed Forward Layer of Decoder Block: 0
Mask Hook fired on Feed Forward Layer of Decoder Block: 1
Mask Hook fired on Feed Forward Layer of Decoder Block: 2
Mask Hook fired on Feed Forward Layer of Decoder Block: 3
Mask Hook fired on Feed Forward Layer of Decoder Block: 4
Mask Hook fired on Feed Forward Layer of Decoder Block: 5
Mask Hook fired on Feed Forward Layer of Decoder Block: 6
Mask Hook fired on Feed Forward Layer of Decoder Block: 7
Mask Hook fired on Feed Forward Layer of Decoder Block: 8
Mask Hook fired on Feed Forward Layer of Decoder Block: 9
Mask Hook fired on Feed Forward Layer of Decoder Block: 10
Mask Hook fired on Feed Forward Layer of Decoder Block: 11
Mask Hook fired

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Streaming output truncated to the last 5000 lines.
Mask Hook fired on Feed Forward Layer of Decoder Block: 21
Random Self Attention Head Masking Hook fired in Decoder Block: 22
Mask Hook fired on Feed Forward Layer of Decoder Block: 22
Random Self Attention Head Masking Hook fired in Decoder Block: 23
Mask Hook fired on Feed Forward Layer of Decoder Block: 23
Random Self Attention Head Masking Hook fired in Decoder Block: 0
Mask Hook fired on Feed Forward Layer of Decoder Block: 0
Random Self Attention Head Masking Hook fired in Decoder Block: 1
Mask Hook fired on Feed Forward Layer of Decoder Block: 1
Random Self Attention Head Masking Hook fired in Decoder Block: 2
Mask Hook fired on Feed Forward Layer of Decoder Block: 2
Random Self Attention Head Masking Hook fired in Decoder Block: 3
Mask Hook fired on Feed Forward Layer of Decoder Block: 3
Random Self Attention Head Masking Hook fired in Decoder Block: 4
Mask Hook fired on Feed Forward Layer of Decoder Block: 4
Random Self Attent

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Streaming output truncated to the last 5000 lines.
Mask Hook fired on Feed Forward Layer of Decoder Block: 21
Random Cross Attention Head Masking Hook fired in Decoder Block: 22
Mask Hook fired on Feed Forward Layer of Decoder Block: 22
Random Cross Attention Head Masking Hook fired in Decoder Block: 23
Mask Hook fired on Feed Forward Layer of Decoder Block: 23
Random Cross Attention Head Masking Hook fired in Decoder Block: 0
Mask Hook fired on Feed Forward Layer of Decoder Block: 0
Random Cross Attention Head Masking Hook fired in Decoder Block: 1
Mask Hook fired on Feed Forward Layer of Decoder Block: 1
Random Cross Attention Head Masking Hook fired in Decoder Block: 2
Mask Hook fired on Feed Forward Layer of Decoder Block: 2
Random Cross Attention Head Masking Hook fired in Decoder Block: 3
Mask Hook fired on Feed Forward Layer of Decoder Block: 3
Random Cross Attention Head Masking Hook fired in Decoder Block: 4
Mask Hook fired on Feed Forward Layer of Decoder Block: 4
Random Cros

...

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Streaming output truncated to the last 5000 lines.
Mask Hook fired on Feed Forward Layer of Decoder Block: 21
Random Self Attention Head Masking Hook fired in Decoder Block: 22
Mask Hook fired on Feed Forward Layer of Decoder Block: 22
Random Self Attention Head Masking Hook fired in Decoder Block: 23
Mask Hook fired on Feed Forward Layer of Decoder Block: 23
Random Self Attention Head Masking Hook fired in Decoder Block: 0
Mask Hook fired on Feed Forward Layer of Decoder Block: 0
Random Self Attention Head Masking Hook fired in Decoder Block: 1
Mask Hook fired on Feed Forward Layer of Decoder Block: 1
Random Self Attention Head Masking Hook fired in Decoder Block: 2
Mask Hook fired on Feed Forward Layer of Decoder Block: 2
Random Self Attention Head Masking Hook fired in Decoder Block: 3
Mask Hook fired on Feed Forward Layer of Decoder Block: 3
Random Self Attention Head Masking Hook fired in Decoder Block: 4
Mask Hook fired on Feed Forward Layer of Decoder Block: 4
Random Self Attent

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Streaming output truncated to the last 5000 lines.
Random Cross Attention Head Masking Hook fired in Decoder Block: 14
Mask Hook fired on Feed Forward Layer of Decoder Block: 14
Random Self Attention Head Masking Hook fired in Decoder Block: 15
Random Cross Attention Head Masking Hook fired in Decoder Block: 15
Mask Hook fired on Feed Forward Layer of Decoder Block: 15
Random Self Attention Head Masking Hook fired in Decoder Block: 16
Random Cross Attention Head Masking Hook fired in Decoder Block: 16
Mask Hook fired on Feed Forward Layer of Decoder Block: 16
Random Self Attention Head Masking Hook fired in Decoder Block: 17
Random Cross Attention Head Masking Hook fired in Decoder Block: 17
Mask Hook fired on Feed Forward Layer of Decoder Block: 17
Random Self Attention Head Masking Hook fired in Decoder Block: 18
Random Cross Attention Head Masking Hook fired in Decoder Block: 18
Mask Hook fired on Feed Forward Layer of Decoder Block: 18
Random Self Attention Head Masking Hook fired 

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# Part 1

- layer dropping (drop encoder block and drop decoder block, one by one, even, odds, from heads towards center, want to see which is more sensible, which collapses first)
- feed forward attention drop (drop encoder block feed forward layers to see if the score remains similar if we eliminate feed forwards, whether it maintains memory or not)

# Part 2

- mask attention heads (mask, 25%, 75%, etc..) and see if accuracy remains high
- if we maintain accuracy above 50% with masked heads, then the distillation process worked well
- mask heads in decoder cross attention to see whether it hallucinates based on some internal weight values (to see how much it counts on structure vs content)

# Part 3

- robustness / sensitivity (noise injection, gaussian noise, compare student vs teacher)
- try on different sizes of inputs and see performance (32, 64, 128, 256 tokens)

# Metrics:

- histogram for cosine similarities (goal would be for all to be 1)
- mean over all cosine similarities
- hard predictions (> 0.5 similarity => predicts 1, else 0) -> obtain some kind of accuracy
- rouge_l
- meteor